# Étape 2 — Réduction de variance

L'erreur standard du Monte Carlo vaut $\sigma_Y / \sqrt{N}$. On ne peut pas améliorer le $\sqrt{N}$, mais on peut construire des estimateurs de **même espérance** et de **variance plus faible**.

Trois méthodes, comparées au Monte Carlo standard :
1. **Variables antithétiques** : on utilise $Z$ et $-Z$.
2. **Variable de contrôle** : on corrige avec $X = e^{-rT} S_T$, dont l'espérance $S_0$ est connue.
3. **Quasi-Monte Carlo randomisé** : suite de Sobol avec décalage aléatoire.

Toutes les méthodes renvoient le même format `(prix, erreur standard, IC)` pour être directement comparables.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, qmc
from scipy.stats import t as student

## 1. Briques de base (reprises de l'étape 1, réorganisées)

In [ ]:
# ---------- Payoffs ----------
def call_payoff(ST, K):
    return np.maximum(ST - K, 0.0)


def put_payoff(ST, K):
    return np.maximum(K - ST, 0.0)


# ---------- Formules fermées de Black-Scholes ----------
def _d1_d2(S0, K, T, r, sigma):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return d1, d1 - sigma * np.sqrt(T)


def bs_call_price(S0, K, T, r, sigma):
    d1, d2 = _d1_d2(S0, K, T, r, sigma)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put_price(S0, K, T, r, sigma):
    d1, d2 = _d1_d2(S0, K, T, r, sigma)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)


# ---------- Simulation : les Z sont maintenant passés en entrée ----------
def terminal_prices(S0, T, r, sigma, Z):
    """S_T exact sous Q, à partir de gaussiennes Z fournies (pseudo-aléatoires, antithétiques, Sobol...)."""
    return S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)


# ---------- Statistiques communes à toutes les méthodes ----------
def summarize(samples, alpha=0.05):
    """
    Prix, erreur standard et IC à partir d'échantillons INDÉPENDANTS et de même loi
    dont l'espérance est le prix recherché.
    Quantile de Student à n-1 degrés de liberté : indispensable pour le QMC (n = 16 répétitions),
    identique au quantile gaussien 1,96 pour les grands échantillons.
    """
    samples = np.asarray(samples)
    price = samples.mean()
    se = samples.std(ddof=1) / np.sqrt(len(samples))
    q = student.ppf(1 - alpha / 2, df=len(samples) - 1)   # quantile de Student (≈ 1,96 si n est grand)
    return price, se, (price - q * se, price + q * se)

## 2. Monte Carlo standard (référence)

$\hat C_N = \frac1N \sum_i Y_i$ avec $Y_i = e^{-rT} h(S_T^{(i)})$.

In [ ]:
def mc_standard(payoff, S0, K, T, r, sigma, n_paths, seed=None):
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(n_paths)
    Y = np.exp(-r * T) * payoff(terminal_prices(S0, T, r, sigma, Z), K)
    return summarize(Y)

## 3. Variables antithétiques

Pour chaque $Z_i$, on calcule $Y_i = h(Z_i)$ et $Y_i' = h(-Z_i)$, puis la moyenne de la paire $\bar Y_i = (Y_i + Y_i')/2$.

$$\mathrm{Var}(\bar Y_i) = \frac{\sigma_Y^2}{2}(1 + \rho), \qquad \rho = \mathrm{Corr}(Y, Y').$$

Gain si $\rho < 0$, garanti pour un payoff monotone en $Z$ (call, put).

⚠️ L'erreur standard se calcule sur les $n$ **moyennes de paires** (indépendantes), pas sur les $2n$ payoffs (corrélés deux à deux).

Coût : $n$ paires = $2n$ évaluations de payoff.

In [ ]:
def mc_antithetic(payoff, S0, K, T, r, sigma, n_pairs, seed=None):
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(n_pairs)
    disc = np.exp(-r * T)
    Y_plus = disc * payoff(terminal_prices(S0, T, r, sigma, Z), K)
    Y_minus = disc * payoff(terminal_prices(S0, T, r, sigma, -Z), K)
    pair_means = 0.5 * (Y_plus + Y_minus)      # n échantillons indépendants
    return summarize(pair_means)

### 3 bis. Pourquoi $\rho \le 0$ pour un payoff monotone

**Le cadre.** On note $g(z) = e^{-rT}\,h\Big(S_0\, e^{(r-\sigma^2/2)T + \sigma\sqrt T\, z}\Big)$ le payoff actualisé vu comme fonction de la gaussienne, $Y = g(Z)$ et $Y' = g(-Z)$. On a montré que, **à budget égal** de $2n$ évaluations de payoff :

$$\operatorname{Var}(\tilde Y_i) = \frac{\sigma_Y^2}{2}(1+\rho), \qquad \frac{\operatorname{Var}\big(\hat C^{\text{std}}_{2n}\big)}{\operatorname{Var}\big(\hat C^{\text{anti}}_n\big)} = \frac{1}{1+\rho}, \qquad \rho = \operatorname{Corr}\big(g(Z), g(-Z)\big)$$

La méthode est utile si et seulement si $\rho < 0$ ; elle **double** la variance dans le pire cas $\rho = 1$. Le résultat suivant garantit qu'on ne perd jamais pour un call ou un put.

**Proposition.** Si $g$ est monotone, alors $\operatorname{Cov}\big(g(Z), g(-Z)\big) \le 0$, donc $\rho \le 0$.

**Preuve.** Supposons $g$ croissante (le cas décroissant est identique). Posons $f(z) = g(z)$, croissante, et $k(z) = g(-z)$, décroissante. Soit $Z'$ une copie indépendante de $Z$. Pour tous réels $z, z'$, les quantités $f(z) - f(z')$ et $k(z) - k(z')$ sont de signes opposés, donc

$$\big(f(Z) - f(Z')\big)\big(k(Z) - k(Z')\big) \le 0.$$

En prenant l'espérance et en développant, comme $Z$ et $Z'$ sont i.i.d. :

$$\mathbb E\big[f(Z)k(Z)\big] - \mathbb E\big[f(Z)\big]\mathbb E\big[k(Z')\big] - \mathbb E\big[f(Z')\big]\mathbb E\big[k(Z)\big] + \mathbb E\big[f(Z')k(Z')\big] = 2\operatorname{Cov}\big(f(Z), k(Z)\big) \le 0,$$

et $\operatorname{Cov}\big(f(Z), k(Z)\big) = \operatorname{Cov}\big(g(Z), g(-Z)\big)$. $\blacksquare$

Le call ($g$ croissante) et le put ($g$ décroissante) ne font donc **jamais** pire que le Monte Carlo standard. Pour un payoff non monotone, il n'y a aucune garantie.

**Ce que la méthode supprime.** Toute fonction se décompose en partie paire et partie impaire :

$$g(z) = \underbrace{\frac{g(z) + g(-z)}{2}}_{g_{\text{paire}}(z)} + \underbrace{\frac{g(z) - g(-z)}{2}}_{g_{\text{impaire}}(z)}, \qquad \tilde Y = g_{\text{paire}}(Z), \qquad \operatorname{Var}(\tilde Y) = \operatorname{Var}\big(g_{\text{paire}}(Z)\big).$$

La méthode élimine **exactement la partie impaire** du payoff :
- payoff presque linéaire en $Z$ (call très dans la monnaie) : gros gain ;
- payoff presque nul sauf pour les grands $Z$ (call très hors de la monnaie) : gain quasi nul ;
- payoff **pair** en $Z$ : rien à supprimer, $\rho = 1$, variance **doublée**.

**Vérification numérique.** On estime $\rho$ et le facteur $\frac{1}{1+\rho}$ pour un call, un put, un payoff non monotone (spread papillon) et un payoff pair en $Z$, $h(S_T) = \big(\ln(S_T/S_0) - (r - \tfrac{\sigma^2}{2})T\big)^2 = \sigma^2 T\, Z^2$.

In [ ]:
def antithetic_check(payoff_of_ST, S0=100.0, T=1.0, r=0.03, sigma=0.2, n=1_000_000, seed=0):
    """Corrélation ρ entre g(Z) et g(-Z), et facteur de réduction de variance théorique 1/(1+ρ)."""
    Z = np.random.default_rng(seed).standard_normal(n)
    disc = np.exp(-r * T)
    Y_plus = disc * payoff_of_ST(terminal_prices(S0, T, r, sigma, Z))
    Y_minus = disc * payoff_of_ST(terminal_prices(S0, T, r, sigma, -Z))
    rho = np.corrcoef(Y_plus, Y_minus)[0, 1]
    return rho, 1 / (1 + rho)


S0_, T_, r_, sigma_ = 100.0, 1.0, 0.03, 0.2
drift_ = (r_ - 0.5 * sigma_**2) * T_
cases = {
    "Call K=100 (croissant)":        lambda ST: np.maximum(ST - 100.0, 0.0),
    "Call K=70 (croissant)":         lambda ST: np.maximum(ST - 70.0, 0.0),
    "Call K=130 (croissant)":        lambda ST: np.maximum(ST - 130.0, 0.0),
    "Put K=100 (décroissant)":       lambda ST: np.maximum(100.0 - ST, 0.0),
    "Papillon 90/100/110 (non monotone)":
        lambda ST: np.maximum(ST - 90, 0) - 2 * np.maximum(ST - 100, 0) + np.maximum(ST - 110, 0),
    "Pair en Z : (ln(S_T/S0) - drift)²": lambda ST: (np.log(ST / S0_) - drift_)**2,
}

print(f"{'Payoff':<38} {'ρ':>8} {'Facteur 1/(1+ρ)':>17}")
for name, h in cases.items():
    rho, factor = antithetic_check(h)
    print(f"{name:<38} {rho:>+8.3f} {factor:>17.2f}")

Les payoffs monotones ont bien $\rho \le 0$ (gain garanti, très fort dans la monnaie, quasi nul hors de la monnaie). Le papillon, non monotone, n'a aucune garantie. Le payoff pair en $Z$ a $\rho = 1$ : les variables antithétiques **doublent** la variance.

## 4. Variable de contrôle

On utilise $X = e^{-rT} S_T$, dont l'espérance est connue : $\mathbb E_{\mathbb Q}[X] = S_0$ (l'actif actualisé est une martingale).

$$Y^{(b)} = Y - b\,(X - S_0), \qquad \mathbb E[Y^{(b)}] = \mathbb E[Y] \text{ pour tout } b.$$

La variance est minimale pour $b^* = \mathrm{Cov}(Y, X) / \mathrm{Var}(X)$ et vaut alors $\sigma_Y^2 (1 - \rho_{XY}^2)$.

$b^*$ est inconnu : on l'estime. Deux options :
- **sur les mêmes simulations** (`n_pilot=0`) : très petit biais en $O(1/N)$, négligeable en pratique ;
- **sur une simulation pilote séparée** (`n_pilot>0`) : estimateur exactement sans biais, pour un petit surcoût.

In [ ]:
def _estimate_b(Y, X):
    """Coefficient optimal b* = Cov(Y, X) / Var(X) (pente de la régression de Y sur X)."""
    cov = np.cov(Y, X, ddof=1)
    return cov[0, 1] / cov[1, 1]


def mc_control_variate(payoff, S0, K, T, r, sigma, n_paths, seed=None, n_pilot=0):
    rng = np.random.default_rng(seed)
    disc = np.exp(-r * T)

    if n_pilot > 0:   # b* estimé sur des simulations indépendantes
        ST_pilot = terminal_prices(S0, T, r, sigma, rng.standard_normal(n_pilot))
        b = _estimate_b(disc * payoff(ST_pilot, K), disc * ST_pilot)

    ST = terminal_prices(S0, T, r, sigma, rng.standard_normal(n_paths))
    Y = disc * payoff(ST, K)
    X = disc * ST                     # E[X] = S0 connu exactement

    if n_pilot == 0:  # b* estimé sur les mêmes simulations
        b = _estimate_b(Y, X)

    Y_cv = Y - b * (X - S0)
    return summarize(Y_cv)

## 5. Quasi-Monte Carlo randomisé (Sobol + décalage aléatoire)

Une suite de Sobol remplit $[0,1]$ bien plus régulièrement que des tirages pseudo-aléatoires : l'erreur est de l'ordre de $(\log N)^d / N$ au lieu de $1/\sqrt N$.

Mais la suite est **déterministe** : pas de TCL, donc pas d'intervalle de confiance. Solution : on la **randomise** par un décalage aléatoire $u \mapsto (u + U) \bmod 1$ avec $U \sim \mathcal U(0,1)$ (rotation de Cranley-Patterson). Chaque point décalé reste uniforme, donc chaque estimation est sans biais, et $R$ décalages indépendants donnent $R$ estimations i.i.d. sur lesquelles on calcule l'erreur standard.

On transforme ensuite les uniformes en gaussiennes par $Z = \Phi^{-1}(u)$ (`norm.ppf`).

Coût : $R \times 2^m$ évaluations de payoff. Les suites de Sobol s'utilisent avec des tailles en puissance de 2.

In [ ]:
def mc_qmc(payoff, S0, K, T, r, sigma, m, n_replications=16, seed=None):
    """
    QMC randomisé : n_replications estimations indépendantes, chacune sur 2^m points de Sobol décalés.
    L'erreur standard est calculée sur les n_replications estimations.
    """
    rng = np.random.default_rng(seed)
    sobol = qmc.Sobol(d=1, scramble=False).random_base2(m)[:, 0]   # 2^m points déterministes
    disc = np.exp(-r * T)

    estimates = np.empty(n_replications)
    for j in range(n_replications):
        u = (sobol + rng.uniform()) % 1.0          # décalage aléatoire
        Z = norm.ppf(u)
        estimates[j] = (disc * payoff(terminal_prices(S0, T, r, sigma, Z), K)).mean()
    return summarize(estimates)

## 6. Comparaison à budget égal

Chaque méthode reçoit le **même nombre d'évaluations de payoff** $N$ :
- standard et variable de contrôle : $N$ trajectoires ;
- antithétique : $N/2$ paires ;
- QMC : $R = 16$ réplications de $N/16$ points.

**Facteur de réduction de variance** = $\mathrm{Var}_{\text{standard}} / \mathrm{Var}_{\text{méthode}}$ à budget égal : c'est le nombre de fois moins de simulations nécessaires pour la même précision.

**Gain d'efficacité** = $(\mathrm{Var} \times \text{temps})_{\text{standard}} / (\mathrm{Var} \times \text{temps})_{\text{méthode}}$ : le vrai critère, qui tient compte du coût de calcul.

In [ ]:
def run_all_methods(payoff, params, N, seed=42, n_reps_qmc=16):
    """Lance les 4 méthodes à budget N et renvoie {nom: (prix, se, temps)}."""
    m_qmc = int(np.log2(N // n_reps_qmc))
    methods = {
        "Standard":          lambda: mc_standard(payoff, **params, n_paths=N, seed=seed),
        "Antithétique":      lambda: mc_antithetic(payoff, **params, n_pairs=N // 2, seed=seed),
        "Var. de contrôle":  lambda: mc_control_variate(payoff, **params, n_paths=N, seed=seed),
        "QMC randomisé":     lambda: mc_qmc(payoff, **params, m=m_qmc, n_replications=n_reps_qmc, seed=seed),
    }
    results = {}
    for name, f in methods.items():
        n_timing = 5
        t0 = time.perf_counter()
        for _ in range(n_timing):
            price, se, _ = f()
        results[name] = (price, se, (time.perf_counter() - t0) / n_timing)
    return results


def print_comparison(results, exact, title):
    print(f"{title} — prix exact : {exact:.5f}\n")
    print(f"{'Méthode':<18} {'Prix':>9} {'Err. std':>9} {'Erreur':>9} {'Réd. var.':>10} {'Temps (ms)':>11} {'Gain effic.':>12}")
    _, se_ref, t_ref = results["Standard"]
    for name, (price, se, t) in results.items():
        vr = se_ref**2 / se**2
        eff = (se_ref**2 * t_ref) / (se**2 * t)
        print(f"{name:<18} {price:>9.5f} {se:>9.5f} {price - exact:>+9.5f} {vr:>10.1f} {1000 * t:>11.2f} {eff:>12.1f}")
    print()


params = dict(S0=100.0, K=100.0, T=1.0, r=0.03, sigma=0.2)
N = 2**18   # 262 144 évaluations de payoff pour chaque méthode

print_comparison(run_all_methods(call_payoff, params, N), bs_call_price(**params), "Call ATM (K = 100)")
print_comparison(run_all_methods(put_payoff, params, N),  bs_put_price(**params),  "Put ATM (K = 100)")

## 7. Influence du strike

Les gains dépendent beaucoup de la moneyness :
- la variable de contrôle $S_T$ est d'autant plus efficace que le call est **dans la monnaie** (le payoff ressemble alors à $S_T - K$, presque linéaire en $S_T$) ;
- l'antithétique perd de son intérêt **hors de la monnaie** (la plupart des paires donnent un payoff nul des deux côtés).

In [ ]:
strikes = [70, 80, 90, 100, 110, 120, 130]
vr = {name: [] for name in ["Antithétique", "Var. de contrôle", "QMC randomisé"]}

for K in strikes:
    p = dict(params, K=float(K))
    res = run_all_methods(call_payoff, p, N)
    se_ref = res["Standard"][1]
    for name in vr:
        vr[name].append(se_ref**2 / res[name][1]**2)

print(f"{'K':>5}" + "".join(f"{name:>18}" for name in vr))
for i, K in enumerate(strikes):
    print(f"{K:>5}" + "".join(f"{vr[name][i]:>18.1f}" for name in vr))

plt.figure(figsize=(7, 5))
for name, marker in zip(vr, ["o", "s", "^"]):
    plt.semilogy(strikes, vr[name], marker + "-", label=name)
plt.axhline(1, color="k", ls="--", lw=1, label="Standard (référence)")
plt.xlabel("Strike K (S0 = 100)")
plt.ylabel("Facteur de réduction de variance (échelle log)")
plt.title("Réduction de variance selon le strike — call européen")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.savefig("reduction_variance_strike.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Convergence : erreur standard en fonction du budget

Les méthodes MC (standard, antithétique, contrôle) gardent la pente $-1/2$ : elles ne changent que la constante $\sigma$. Le QMC, lui, a une pente plus raide, proche de $-1$.

In [ ]:
budgets = [2**k for k in range(10, 21)]
se_curves = {name: [] for name in ["Standard", "Antithétique", "Var. de contrôle", "QMC randomisé"]}

for n in budgets:
    se_curves["Standard"].append(mc_standard(call_payoff, **params, n_paths=n, seed=1)[1])
    se_curves["Antithétique"].append(mc_antithetic(call_payoff, **params, n_pairs=n // 2, seed=1)[1])
    se_curves["Var. de contrôle"].append(mc_control_variate(call_payoff, **params, n_paths=n, seed=1)[1])
    se_curves["QMC randomisé"].append(mc_qmc(call_payoff, **params, m=int(np.log2(n // 16)), n_replications=16, seed=1)[1])

b = np.array(budgets, dtype=float)
plt.figure(figsize=(7, 5))
for name, marker in zip(se_curves, ["o", "s", "^", "D"]):
    plt.loglog(b, se_curves[name], marker + "-", label=name)
plt.loglog(b, se_curves["Standard"][0] * np.sqrt(b[0] / b), "k--", lw=1, label=r"Pente $-1/2$")
plt.loglog(b, se_curves["QMC randomisé"][0] * (b[0] / b), "k:", lw=1, label=r"Pente $-1$")
plt.xlabel("Budget N (évaluations de payoff)")
plt.ylabel("Erreur standard")
plt.title("Convergence des méthodes — call ATM")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.savefig("convergence_methodes.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Vérification : biais de la variable de contrôle

On compare $b^*$ estimé sur les mêmes simulations et sur une simulation pilote indépendante. Les deux prix doivent être compatibles avec la formule fermée.

In [ ]:
exact = bs_call_price(**params)
for label, n_pilot in [("b* sur les mêmes simulations", 0), ("b* sur un pilote de 10 000", 10_000)]:
    price, se, (lo, hi) = mc_control_variate(call_payoff, **params, n_paths=N, seed=42, n_pilot=n_pilot)
    print(f"{label:<32} prix = {price:.5f}  err. std = {se:.5f}  IC = [{lo:.5f}, {hi:.5f}]  exact dans IC : {lo <= exact <= hi}")